# 98 — Skeleton-of-Thought
## Parallel LLM expansion with LangGraph `Send()` fan-out
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/98-skeleton-of-thought/skeleton_of_thought_workbook.ipynb)

**The problem:** LLMs generate tokens sequentially — one at a time. When you ask for a long, structured answer, the model writes the entire response word-by-word. If the answer has five independent sections, the fifth section can't start until the first four are fully written. That's avoidable latency.

**The technique — Skeleton-of-Thought (SoT):** Proposed by Ning et al. (2023), SoT splits generation into two phases:
1. **Skeleton phase** — one fast LLM call produces a numbered outline (the "skeleton")
2. **Expansion phase** — each skeleton point is expanded *in parallel*, each with its own independent LLM call

**Why it matters:** Wall-clock time shrinks from `O(N × expand_time)` to `O(skeleton_time + expand_time)` — essentially constant regardless of how many points the skeleton contains. This notebook implements SoT using LangGraph's `Send()` primitive for true concurrent fan-out.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — Sequential vs. parallel generation, SoT pattern |
| 2 | **Setup** — Install deps, configure API key |
| 3 | **State Design** — `SoTState`, `PointState`, and Annotated reducers |
| 4 | **Prompts & Tools** — Skeleton and expansion prompts |
| 5 | **Graph Nodes** — `make_skeleton`, `dispatch`, `expand`, `concatenate` |
| 6 | **Graph Wiring** — `conditional_edges` + `Send()` fan-out |
| 7 | **End-to-End Run** — Full workflow on real questions |
| 8 | **Timing Analysis** — Measuring parallelism benefit |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `langchain-openai`, `langgraph`, `python-dotenv`

### Key References
> Ning et al. 2023 — *Skeleton-of-Thought: Large Language Models Can Do Parallel Decoding*  
> [arxiv.org/abs/2307.15337](https://arxiv.org/abs/2307.15337)
>
> LangGraph `Send()` docs — [langchain-ai.github.io/langgraph](https://langchain-ai.github.io/langgraph/)

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain-openai", "langgraph", "python-dotenv"
    ])
    print("Colab install complete.")
else:
    print("Local — skipping install. Ensure deps are in your venv.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
if not (bool(key) and key.startswith('sk-')):
    raise RuntimeError("OPENAI_API_KEY is required for the live skeleton-of-thought cells.")
print("API key ready: True")

---
## Part 1 — Sequential vs. Parallel Generation

### The sequential baseline

When you ask an LLM "Explain X with 5 key points", generation looks like this:

```
Time →
┌────────────────────────────────────────────────────────────────┐
│ Point 1 ░░░░░░ | Point 2 ░░░░░░ | Point 3 ░░░░░░ | ...       │
└────────────────────────────────────────────────────────────────┘
Total time = sum of all points
```

### The SoT approach

```
Time →
┌─────────────┐
│ Skeleton    │  (one fast call — outline only)
└──────┬──────┘
       │ fan-out via Send()
       ├──────────────────────────────────────
       │ Point 1 ░░░░░░                        │ all expand calls
       │ Point 2 ░░░░░░                        │ fire at the same
       │ Point 3 ░░░░░░                        │ time
       │ Point 4 ░░░░░░                        │
       │ Point 5 ░░░░░░                        │
       └──────────────────────────────────────
┌──────────────┐
│ Concatenate  │  (sort by idx, join)
└──────────────┘
Total time ≈ skeleton_time + max(expand_time)  ← constant!
```

### When does SoT help?

| Condition | SoT benefit |
|-----------|-------------|
| Many independent sections | High — each runs in parallel |
| Sections depend on each other | Low — dependencies require serialization |
| Short answers (1-2 points) | Marginal — overhead may outweigh gain |
| Long technical explanations | High — classic use case |
| Creative writing (narrative flow) | Low — coherence requires sequential context |

> **Paper finding:** Ning et al. report up to 2.39× speedup on benchmarks with no quality degradation on fact-based questions.

---
## Part 2 — State Design

LangGraph manages state as a `TypedDict`. For SoT we need two state types:

### `SoTState` — the main graph state

| Field | Type | Role |
|-------|------|------|
| `question` | `str` | The input question |
| `skeleton` | `list[str]` | Numbered outline points (from `make_skeleton`) |
| `expansions` | `Annotated[list[dict], operator.add]` | Accumulated expansions from parallel calls |
| `answer` | `str` | Final joined answer |

### `PointState` — state for each parallel expand node

| Field | Type | Role |
|-------|------|------|
| `question` | `str` | Forwarded for context |
| `point` | `str` | The specific skeleton point to expand |
| `idx` | `int` | Position in skeleton — critical for ordering |

### The `Annotated` reducer — why it matters

Without a reducer, if two parallel nodes both write to `expansions`, the second write would overwrite the first. The `Annotated[list[dict], operator.add]` annotation tells LangGraph to **accumulate** (append) instead of overwrite:

```python
# Without reducer:  state["expansions"] = new_value   (OVERWRITES)
# With operator.add: state["expansions"] += new_value  (ACCUMULATES)
```

Each `expand` node returns `{"expansions": [{"idx": i, "point": p, "content": c}]}` — a list with one item. The reducer concatenates all these lists as parallel nodes complete.

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict


class SoTState(TypedDict):
    question:   str
    skeleton:   list[str]
    # operator.add accumulates one {idx, point, content} dict per parallel expand call.
    # idx is required — parallel execution returns in arbitrary order.
    expansions: Annotated[list[dict], operator.add]
    answer:     str


class PointState(TypedDict):
    question: str
    point:    str
    idx:      int


# Verify the TypedDicts are valid
print("SoTState fields:", list(SoTState.__annotations__.keys()))
print("PointState fields:", list(PointState.__annotations__.keys()))

# Demonstrate the reducer behavior
demo_accumulate = operator.add([{"idx": 0, "content": "A"}], [{"idx": 1, "content": "B"}])
print("\noperator.add on two lists:", demo_accumulate)
print("→ Both items preserved — this is the reducer behavior LangGraph uses.")

---
## Part 3 — Prompts

SoT uses two prompts with very different objectives:

### Skeleton prompt — structure first, content second

The key constraint is `Output ONLY the skeleton: one numbered point per line, no elaboration`. This keeps the first call fast and ensures each point is parseable. We parse by splitting on newlines.

### Expansion prompt — focused, context-aware

Each expand call receives:
- The original question (for semantic context)
- Its specific point only (not the full skeleton)

This is a deliberate trade-off: each expand call works without awareness of what other points say, which is what enables true parallelism. For questions where sections are truly independent, this works well. For questions where section 3 logically builds on section 2, you'd need a different approach.

In [ ]:
# Two questions that produce 4-5 skeleton points — enough to show meaningful parallelism.
QUESTIONS = [
    "What are the key differences between supervised, unsupervised, and reinforcement learning?",
    "How does a transformer neural network work?",
]

SKELETON_PROMPT = """\
Generate a numbered skeleton outline for a clear answer to the question below.
Output ONLY the skeleton: one numbered point per line, no elaboration, 4-5 points.

Question: {question}"""

# Each expand call receives only its own point — parallel calls have no shared context.
EXPAND_PROMPT = """\
Expand the following skeleton point into 2-3 concise, informative sentences.
Stay focused on this point only.

Question context: {question}
Point: {point}"""


# Inspect what a formatted skeleton prompt looks like
sample_q = QUESTIONS[0]
print("=== Skeleton Prompt (formatted) ===")
print(SKELETON_PROMPT.format(question=sample_q))

print("\n=== Expand Prompt (formatted for one point) ===")
sample_point = "1. Supervised learning requires labeled training data"
print(EXPAND_PROMPT.format(question=sample_q, point=sample_point))

---
## Part 4 — Graph Nodes

The SoT graph has four logical components:

### `make_skeleton` — the serial bottleneck
One LLM call. Takes the question, returns a list of numbered strings. This is the only sequential step — everything after this fires in parallel.

### `dispatch` — the fan-out function
Returns a list of `Send()` objects — one per skeleton point. Each `Send()` specifies:
- The target node name (`"expand"`)
- The `PointState` to pass it (question, point text, and crucially `idx`)

LangGraph treats `dispatch` as a conditional edge function. When it returns `list[Send]`, LangGraph fires all those edges concurrently.

### `expand` — parallel leaf nodes
Each instance receives its own `PointState`. They all run at the same time. The return value is `{"expansions": [{"idx": ..., "point": ..., "content": ...}]}` — a single-item list that the reducer will accumulate.

### `concatenate` — the reducer
After all parallel `expand` calls complete, `concatenate` receives the accumulated `expansions` list. It sorts by `idx` to restore correct ordering (parallel completion is non-deterministic), then joins everything into the final answer.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.constants import Send

_llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)


def make_skeleton(state: SoTState) -> dict:
    """Single LLM call that determines structure — the serial bottleneck before parallelism."""
    reply = _llm.invoke([
        {"role": "user", "content": SKELETON_PROMPT.format(question=state["question"])}
    ])
    points = [l.strip() for l in reply.content.strip().splitlines() if l.strip()]
    return {"skeleton": points}


def dispatch(state: SoTState) -> list[Send]:
    """Fan out one expand call per skeleton point — all fire concurrently."""
    return [
        Send("expand", {"question": state["question"], "point": p, "idx": i})
        for i, p in enumerate(state["skeleton"])
    ]


def expand(state: PointState) -> dict:
    """Expand one point into prose; tag with idx so concatenate can restore order."""
    reply = _llm.invoke([
        {"role": "user", "content": EXPAND_PROMPT.format(
            question=state["question"], point=state["point"]
        )}
    ])
    return {"expansions": [{"idx": state["idx"], "point": state["point"], "content": reply.content.strip()}]}


def concatenate(state: SoTState) -> dict:
    """Sort by idx to undo any out-of-order arrival, then join into the final answer."""
    ordered = sorted(state["expansions"], key=lambda e: e["idx"])
    answer  = "\n\n".join(f"{e['point']}\n{e['content']}" for e in ordered)
    return {"answer": answer}


print("All node functions defined:")
print("  make_skeleton — 1 LLM call, returns skeleton list")
print("  dispatch      — returns list[Send], fires parallel expand nodes")
print("  expand        — 1 LLM call per point, runs concurrently")
print("  concatenate   — sorts by idx, joins into final answer")

---
## Part 5 — Graph Wiring

The graph topology is what makes parallelism possible. Let's trace each edge:

```
START
  │
  ▼
make_skeleton  ←── one LLM call, produces skeleton list
  │
  │  conditional_edges("make_skeleton", dispatch, ["expand"])
  │  dispatch() returns list[Send("expand", PointState)]
  │
  ├─────────────────────────────────────────────────┐
  │                                                 │
  ▼                                                 ▼
expand(idx=0)     expand(idx=1)     expand(idx=2)  ...  ← all concurrent
  │                    │                │
  └────────────────────┴────────────────┘
                       │
                       ▼
                 concatenate  ←── waits for ALL expand calls to finish
                       │
                       ▼
                      END
```

### Key API: `add_conditional_edges`

```python
g.add_conditional_edges("make_skeleton", dispatch, ["expand"])
```

- Source node: `"make_skeleton"`
- Router function: `dispatch` — called with the current state, returns `list[Send]`
- Possible destinations: `["expand"]` — tells LangGraph which nodes the router may target

When `dispatch` returns a `list[Send]`, LangGraph fires all of them as concurrent branches. LangGraph's runtime waits for all branches to complete before advancing past `expand`.

In [ ]:
from langgraph.graph import StateGraph, START, END


def create_workflow():
    g = StateGraph(SoTState)
    g.add_node("make_skeleton", make_skeleton)
    g.add_node("expand",        expand)
    g.add_node("concatenate",   concatenate)
    g.add_edge(START, "make_skeleton")
    g.add_conditional_edges("make_skeleton", dispatch, ["expand"])
    g.add_edge("expand",      "concatenate")
    g.add_edge("concatenate", END)
    return g.compile()


app = create_workflow()
print("Workflow compiled successfully.")
print("\nGraph nodes:", list(app.nodes.keys()))

---
## Part 6 — Visualising the Graph

LangGraph can render the graph structure as a Mermaid diagram. This helps confirm the fan-out topology is wired correctly before running.

In [ ]:
try:
    from IPython.display import Image, display
    img_bytes = app.get_graph().draw_mermaid_png()
    display(Image(img_bytes))
    print("Graph rendered above.")
except Exception as e:
    print(f"Graph rendering skipped ({type(e).__name__}). Mermaid source:")
    print(app.get_graph().draw_mermaid())

---
## Part 7 — The `idx` Ordering Problem

This is an important subtlety: parallel LLM calls complete in **non-deterministic order**. If Point 3 happens to be shorter than Point 1, it will finish first and arrive in `expansions` earlier. Without the `idx` field, `concatenate` would produce a scrambled answer.

### The solution: tag-and-sort

1. `dispatch` assigns `idx = i` to each `Send()` — this records the intended position
2. Each `expand` call stores `idx` in its return dict: `{"idx": state["idx"], ...}`
3. `concatenate` sorts by `idx` before joining

This is a general pattern for any fan-out workflow where order matters.

In [ ]:
# Demonstrate why idx matters — simulate out-of-order arrival
out_of_order_expansions = [
    {"idx": 2, "point": "3. Reinforcement learning", "content": "RL content..."},
    {"idx": 0, "point": "1. Supervised learning", "content": "SL content..."},
    {"idx": 1, "point": "2. Unsupervised learning", "content": "UL content..."},
]

print("Raw arrival order (non-deterministic):")
for e in out_of_order_expansions:
    print(f"  idx={e['idx']} → {e['point'][:30]}...")

# What concatenate does
ordered = sorted(out_of_order_expansions, key=lambda e: e["idx"])
print("\nAfter sort by idx (correct order):")
for e in ordered:
    print(f"  idx={e['idx']} → {e['point'][:30]}...")

print("\n→ Tag-and-sort restores correct order regardless of completion order.")

---
## Part 8 — End-to-End Run

Now we run the full SoT workflow on both questions. Watch the output:
- `skeleton` shows the outline from the first LLM call
- `answer` shows the parallel-expanded full response
- The point count tells you how many concurrent LLM calls fired

In [ ]:
print("=== 98 · Skeleton-of-Thought ===")
print("Serial: one LLM call builds the outline")
print("Parallel: each point expanded concurrently via Send()")
print("Ref: Ning et al. 2023 — arxiv.org/abs/2307.15337\n")

for question in QUESTIONS:
    print(f"Q: {question}\n")

    result = app.invoke({
        "question":   question,
        "skeleton":   [],
        "expansions": [],
        "answer":     "",
    })

    print("Skeleton (generated in one call):")
    for point in result["skeleton"]:
        print(f"  {point}")
    print(f"\nExpanded in parallel ({len(result['skeleton'])} concurrent LLM calls):")
    print("-" * 60)
    print(result["answer"])
    print("=" * 70)
    print()

print("Note: wall-clock time ≈ skeleton call + ONE expand call (not N).")
print("LangGraph fires all expand nodes concurrently; idx sorts the result.")

---
## Part 9 — Timing Analysis

Let's measure the actual time savings. We'll time:
1. The SoT approach (skeleton + parallel expansion)
2. A simulated sequential baseline (skeleton + N sequential expansions)

Then compare wall-clock times.

In [ ]:
import time

test_question = "What are the main components of a modern operating system?"

# --- SoT (parallel) ---
print("Running SoT (parallel)...")
t0 = time.perf_counter()
sot_result = app.invoke({
    "question":   test_question,
    "skeleton":   [],
    "expansions": [],
    "answer":     "",
})
sot_time = time.perf_counter() - t0
n_points = len(sot_result["skeleton"])

print(f"SoT time: {sot_time:.2f}s for {n_points} points")

# --- Sequential baseline (N individual calls, not concurrent) ---
print("\nRunning sequential baseline...")
t1 = time.perf_counter()

# Reuse the skeleton from above, just expand sequentially
sequential_expansions = []
for i, point in enumerate(sot_result["skeleton"]):
    reply = _llm.invoke([
        {"role": "user", "content": EXPAND_PROMPT.format(
            question=test_question, point=point
        )}
    ])
    sequential_expansions.append(reply.content.strip())

seq_time = time.perf_counter() - t1
print(f"Sequential time: {seq_time:.2f}s for {n_points} points")

speedup = seq_time / sot_time if sot_time > 0 else float('inf')
print(f"\nSpeedup factor: {speedup:.2f}x")
print(f"Time saved: {seq_time - sot_time:.2f}s")
print(f"\nNote: SoT time includes the skeleton call. As N grows, speedup approaches N.")

---
## Part 10 — Inspecting Individual Expansions

Let's look at the raw `expansions` list before concatenation to see what each parallel call returned and confirm the `idx` tagging.

In [ ]:
# Use the result from the timing run above
print(f"Question: {test_question}\n")
print(f"Raw expansions list ({len(sot_result['expansions'])} items):")
print("(Note: list order may not match skeleton order — that's expected)")
print()

for exp in sot_result["expansions"]:
    print(f"idx={exp['idx']} | {exp['point'][:50]}")
    print(f"  → {exp['content'][:100]}...")
    print()

print("After sort by idx, these become the ordered answer.")

---
## Part 11 — Design Variants and Trade-offs

### Variant 1: Skeleton with explicit section names

Instead of `"1. Key differences"`, prompt for `"## Section Title | brief description"` so you can use the section title as a markdown header in the expansion.

### Variant 2: Quality-gated expansion

After expanding, run a critique pass on each section in parallel (another fan-out), then revise. Now you have three waves: skeleton → expand → critique (all three waves are themselves parallelised within each wave).

### Variant 3: Adaptive skeleton size

Ask the LLM to choose the number of points based on question complexity, rather than constraining to 4-5.

### Variant 4: Cross-section awareness

Pass the full skeleton as context to each expand call. This breaks the independence assumption but may improve coherence for narrative questions.

### Trade-off summary

| Factor | SoT strength | SoT weakness |
|--------|-------------|---------------|
| Latency | Excellent — parallel | Extra skeleton call overhead |
| Cost | Same total tokens | Slightly more (skeleton overhead) |
| Coherence | Good for factual Q&A | Weaker for narrative flow |
| Ordering | Solved by idx | Must implement correctly |
| Scalability | Constant time with N | API rate limits at large N |

In [ ]:
# Variant demo: pass full skeleton as context to each expand call
EXPAND_WITH_CONTEXT_PROMPT = """\
You are writing section {idx_plus_1} of {total} in a structured answer.
The full outline is:
{full_skeleton}

Expand ONLY the following point into 2-3 concise sentences:
Point: {point}

Question context: {question}"""


def expand_with_context(state: PointState) -> dict:
    """Variant: each expand call sees the full skeleton for coherence."""
    # Note: we'd need to pass full_skeleton and total in state for this to work
    # This is a design sketch — full_skeleton would need to be in PointState
    print(f"  [Would expand point {state['idx']} with full skeleton context]")
    return {"expansions": [{"idx": state["idx"], "point": state["point"], "content": "[expanded]"}]}


print("Variant: expand_with_context()")
print("  Pros: Better cross-section coherence")
print("  Cons: More tokens per call; still parallel but with larger prompts")
print("  When to use: Narrative essays, tutorials where sections reference each other")
print()
print("Original expand() — no cross-section context")
print("  Pros: Minimum tokens, true independence")
print("  When to use: Factual comparisons, technical explanations, reference docs")

---
## Part 12 — Custom Question

Try SoT on any question. Edit `your_question` below and run the cell.

In [ ]:
your_question = "What are the key principles of good software architecture?"

print(f"Q: {your_question}\n")

custom_result = app.invoke({
    "question":   your_question,
    "skeleton":   [],
    "expansions": [],
    "answer":     "",
})

print(f"Skeleton ({len(custom_result['skeleton'])} points — each will expand in parallel):")
for point in custom_result["skeleton"]:
    print(f"  {point}")

print("\n" + "=" * 60)
print("FULL ANSWER:")
print("=" * 60)
print(custom_result["answer"])

---
## Part 13 — Streaming SoT

LangGraph supports streaming events. We can use `app.stream()` to observe skeleton and expansion events as they happen — useful for debugging and understanding the execution order.

In [ ]:
stream_question = "What are the benefits of functional programming?"

print(f"Q: {stream_question}")
print("Streaming graph events:\n")

for chunk in app.stream(
    {
        "question":   stream_question,
        "skeleton":   [],
        "expansions": [],
        "answer":     "",
    },
    stream_mode="updates",
):
    for node_name, node_output in chunk.items():
        if node_name == "make_skeleton":
            print(f"[make_skeleton] Skeleton ready: {len(node_output.get('skeleton', []))} points")
        elif node_name == "expand":
            exps = node_output.get("expansions", [])
            if exps:
                print(f"[expand] idx={exps[0]['idx']} arrived — {exps[0]['point'][:40]}...")
        elif node_name == "concatenate":
            print(f"[concatenate] Final answer assembled ({len(node_output.get('answer', ''))} chars)")

print("\nObserve: expand events may arrive in any order (non-deterministic).")

---
## Exercises

### Exercise 1 — Configurable skeleton size

Modify `SKELETON_PROMPT` and the initial state to accept a `max_points: int` parameter (default 5). The prompt should include it as a constraint. Run SoT with `max_points=3` and `max_points=7` on the same question and compare the results.

### Exercise 2 — Critique wave

After the expansion phase, add a fourth graph node `critique` that runs in parallel (another `Send()` fan-out) to evaluate each expanded section for accuracy and completeness. Return a `{"idx": ..., "critique": "..."}` dict. Then add a fifth node `revise` that fans out again to revise each section based on its critique. Wire it all together.

### Exercise 3 — Selective expansion

Modify `dispatch` so that skeleton points shorter than 30 characters are passed through unchanged (no LLM call needed), while longer points go through `expand`. This demonstrates conditional fan-out where some branches skip expensive processing.

In [ ]:
# =============================================================================
# ANSWER KEY — Exercise 1: Configurable skeleton size
# =============================================================================

SKELETON_PROMPT_CONFIGURABLE = """\
Generate a numbered skeleton outline for a clear answer to the question below.
Output ONLY the skeleton: one numbered point per line, no elaboration.
Use exactly {max_points} points.

Question: {question}"""


def make_skeleton_configurable(state: SoTState) -> dict:
    max_points = state.get("max_points", 5)  # default 5
    reply = _llm.invoke([
        {"role": "user", "content": SKELETON_PROMPT_CONFIGURABLE.format(
            question=state["question"],
            max_points=max_points
        )}
    ])
    points = [l.strip() for l in reply.content.strip().splitlines() if l.strip()]
    return {"skeleton": points}


# Add max_points to state
class SoTStateConfigurable(TypedDict):
    question:   str
    max_points: int
    skeleton:   list[str]
    expansions: Annotated[list[dict], operator.add]
    answer:     str


def create_configurable_workflow():
    g = StateGraph(SoTStateConfigurable)
    g.add_node("make_skeleton", make_skeleton_configurable)
    g.add_node("expand",        expand)
    g.add_node("concatenate",   concatenate)
    g.add_edge(START, "make_skeleton")
    g.add_conditional_edges("make_skeleton", dispatch, ["expand"])
    g.add_edge("expand",      "concatenate")
    g.add_edge("concatenate", END)
    return g.compile()


config_app = create_configurable_workflow()
test_q = "What makes Python a popular programming language?"

for max_pts in [3, 5]:
    r = config_app.invoke({
        "question": test_q,
        "max_points": max_pts,
        "skeleton": [],
        "expansions": [],
        "answer": "",
    })
    print(f"max_points={max_pts} → got {len(r['skeleton'])} points: {[p[:30]+'...' for p in r['skeleton']]}")

In [ ]:
# =============================================================================
# ANSWER KEY — Exercise 2: Critique wave (sketch — two-wave fan-out)
# =============================================================================

CRITIQUE_PROMPT = """\
Critique the following section expansion for accuracy and completeness in 1 sentence.
Be specific about what is missing or could be improved.

Point: {point}
Expansion: {content}"""

REVISE_PROMPT = """\
Revise the following expansion based on the critique. Keep it to 2-3 sentences.

Original expansion: {content}
Critique: {critique}"""


class ExpandedPointState(TypedDict):
    idx:      int
    point:    str
    content:  str


class SoTCritiqueState(TypedDict):
    question:   str
    skeleton:   list[str]
    expansions: Annotated[list[dict], operator.add]
    critiqued:  Annotated[list[dict], operator.add]  # {idx, point, content, critique, revised}
    answer:     str


def critique_dispatch(state: SoTCritiqueState) -> list[Send]:
    """Second fan-out: one critique call per expansion."""
    return [
        Send("critique", {"idx": e["idx"], "point": e["point"], "content": e["content"]})
        for e in sorted(state["expansions"], key=lambda e: e["idx"])
    ]


def critique(state: ExpandedPointState) -> dict:
    """Critique one expanded section, then revise it inline."""
    critique_reply = _llm.invoke([
        {"role": "user", "content": CRITIQUE_PROMPT.format(
            point=state["point"], content=state["content"]
        )}
    ])
    critique_text = critique_reply.content.strip()

    revise_reply = _llm.invoke([
        {"role": "user", "content": REVISE_PROMPT.format(
            content=state["content"], critique=critique_text
        )}
    ])
    return {"critiqued": [{
        "idx":      state["idx"],
        "point":    state["point"],
        "content":  state["content"],
        "critique": critique_text,
        "revised":  revise_reply.content.strip(),
    }]}


def concatenate_revised(state: SoTCritiqueState) -> dict:
    ordered = sorted(state["critiqued"], key=lambda e: e["idx"])
    answer  = "\n\n".join(f"{e['point']}\n{e['revised']}" for e in ordered)
    return {"answer": answer}


print("Exercise 2 answer key: critique wave structure defined.")
print("Graph would be: make_skeleton → expand (×N) → critique (×N) → concatenate_revised")
print("Two Send() fan-outs: dispatch (expand) and critique_dispatch (critique)")
print("\nRunning a minimal demo...")

# Build the critique workflow
cg = StateGraph(SoTCritiqueState)
cg.add_node("make_skeleton",       make_skeleton)
cg.add_node("expand",              expand)
cg.add_node("critique",            critique)
cg.add_node("concatenate_revised", concatenate_revised)
cg.add_edge(START, "make_skeleton")
cg.add_conditional_edges("make_skeleton", dispatch, ["expand"])
cg.add_conditional_edges("expand",        critique_dispatch, ["critique"])
cg.add_edge("critique",            "concatenate_revised")
cg.add_edge("concatenate_revised", END)
critique_app = cg.compile()

crit_result = critique_app.invoke({
    "question":   "What are the benefits of unit testing?",
    "skeleton":   [],
    "expansions": [],
    "critiqued":  [],
    "answer":     "",
})

print(f"\nCritique wave complete. {len(crit_result['critiqued'])} sections critiqued and revised.")
print("\nFirst section:")
first = sorted(crit_result["critiqued"], key=lambda e: e["idx"])[0]
print(f"  Point:    {first['point']}")
print(f"  Original: {first['content'][:80]}...")
print(f"  Critique: {first['critique'][:80]}...")
print(f"  Revised:  {first['revised'][:80]}...")

In [ ]:
# =============================================================================
# ANSWER KEY — Exercise 3: Selective expansion (skip short points)
# =============================================================================

SHORT_POINT_THRESHOLD = 30  # characters


def dispatch_selective(state: SoTState) -> list[Send | dict]:
    """
    Fan out only for points longer than threshold.
    Short points are passed through to concatenate directly.

    Note: In LangGraph, you can mix Send() with direct state updates
    by returning state updates for short points and Send() for long ones.
    Here we use a simpler approach: always Send to expand, but expand
    checks length and skips the LLM call if the point is short.
    """
    return [
        Send("expand_selective", {"question": state["question"], "point": p, "idx": i})
        for i, p in enumerate(state["skeleton"])
    ]


def expand_selective(state: PointState) -> dict:
    """Skip LLM call for short points; expand long ones."""
    # Strip the number prefix (e.g. "1. ") for length check
    point_text = state["point"]
    text_without_number = point_text.split(".", 1)[-1].strip() if "." in point_text else point_text

    if len(text_without_number) < SHORT_POINT_THRESHOLD:
        # Pass through without LLM call
        content = f"[Short point — passed through]: {point_text}"
        print(f"  Skipping LLM for short point idx={state['idx']}: {point_text[:40]}")
    else:
        reply = _llm.invoke([
            {"role": "user", "content": EXPAND_PROMPT.format(
                question=state["question"], point=point_text
            )}
        ])
        content = reply.content.strip()
        print(f"  Expanded idx={state['idx']}: {point_text[:40]}...")

    return {"expansions": [{"idx": state["idx"], "point": point_text, "content": content}]}


# Build selective workflow
sg = StateGraph(SoTState)
sg.add_node("make_skeleton",  make_skeleton)
sg.add_node("expand_selective", expand_selective)
sg.add_node("concatenate",    concatenate)
sg.add_edge(START, "make_skeleton")
sg.add_conditional_edges("make_skeleton", dispatch_selective, ["expand_selective"])
sg.add_edge("expand_selective", "concatenate")
sg.add_edge("concatenate",       END)
selective_app = sg.compile()

print(f"Selective expansion (threshold: {SHORT_POINT_THRESHOLD} chars)")
print("Points shorter than threshold skip the LLM expand call.\n")

sel_result = selective_app.invoke({
    "question":   "What are the core Python data types?",
    "skeleton":   [],
    "expansions": [],
    "answer":     "",
})

print(f"\nResult: {len(sel_result['skeleton'])} points processed.")

---
## Summary — What You Built

You implemented and explored the **Skeleton-of-Thought** pattern:

| Component | What it does | Key detail |
|-----------|-------------|------------|
| `SoTState` | Holds graph state with `Annotated` reducer | `operator.add` accumulates parallel results |
| `make_skeleton` | Serial LLM call → outline | The only sequential bottleneck |
| `dispatch` | Returns `list[Send]` | Triggers concurrent fan-out |
| `expand` | One LLM call per point | All instances run simultaneously |
| `concatenate` | Sorts by `idx`, joins | Restores correct order after parallel arrival |
| `create_workflow` | Wires graph with `conditional_edges` | The `["expand"]` list registers valid targets |

### Core insight

**Total wall-clock time** = `skeleton_latency + max(expand_latency)` regardless of N. This is the key result from Ning et al. 2023 — parallel decoding doesn't degrade quality on fact-based Q&A while delivering significant latency improvements.

### The `idx` pattern

Tag every parallel output with its intended position. Sort after collection. This is the canonical solution to ordering non-deterministic parallel completions in LangGraph.

---

**Workshop complete. Next: example 99 — [coming soon].**

> Reference: Ning et al. (2023). *Skeleton-of-Thought: Large Language Models Can Do Parallel Decoding.*  
> [arxiv.org/abs/2307.15337](https://arxiv.org/abs/2307.15337)